# GANForge -- Progressive GAN (Standalone Notebook)

**Stable high-resolution generation via progressive growing**

This is a **self-contained extract** of the Progressive GAN section from
the [GANForge Master Notebook](GANForge_Master_Notebook.ipynb) -- every
shared setup utility it needs (imports, device selection, seeding,
`weights_init`, `show_images`, checkpointing, etc.) is included below in
Section 0, so this notebook runs **top to bottom on its own**, in Colab
or locally, without needing any other file.

> **⚠️ Compute note.** See the Section 0 GPU warning below. Heavier
> architectures (Progressive GAN, StyleGAN2, BigGAN, StackGAN, GameGAN)
> include a lightweight from-scratch training loop sized for a single
> GPU (or CPU, slowly) plus, where realistically available, code to load
> strong pretrained weights for inference.

<a id="0"></a>
## 0. Global Setup & Utilities

Everything shared across all twelve GANs lives here: installs, imports,
device selection, seeding, and small helper functions (`weights_init`,
`show_images`, `save_checkpoint`, `Logger`, an `EMA` helper, and a spectral
-norm-aware conv wrapper). Re-using these keeps every later section short
and focused on what makes *that* GAN different.

**🚀 Pro Tip.** In Google Colab: `Runtime -> Change runtime type -> T4 GPU`
before running anything. Everything here auto-detects CUDA and degrades
gracefully to CPU (slow but functional) if none is found.

In [ ]:
# Colab-only: uncomment to install packages not preinstalled in Colab.
# All of these are optional -- every dataset loader below has an offline
# synthetic fallback, so the notebook still runs without them.
# !pip install -q torchaudio datasets soundfile pillow-simd torchmetrics
# !pip install -q pytorch-fid clean-fid


In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import os
import io
import math
import time
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.transforms as T
import torchvision.utils as vutils

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    # Minimal shim so the notebook still runs if tqdm is somehow missing.
    def tqdm(iterable, *args, **kwargs):
        return iterable

warnings.filterwarnings("ignore")  # GAN training is noisy (deprecation spam); keep output readable

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)


In [ ]:
# ---------------------------------------------------------------------------
# Device setup
# ---------------------------------------------------------------------------
# We centralize device selection so every model/tensor in the notebook is
# created consistently. All training loops below check DEVICE.type == "cuda"
# to decide whether to use mixed precision (torch.cuda.amp), which roughly
# halves memory use and often speeds up training 2-3x on modern GPUs.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found -- running on CPU. Reduce EPOCHS/BATCH_SIZE in "
          "every section for reasonable runtimes.")


In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
# GANs are notoriously sensitive to initialization -- fixing every RNG makes
# results reproducible across reruns, which matters a lot when you are
# debugging "why did my generator collapse this time".
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
# ---------------------------------------------------------------------------
# Shared helper functions used across every GAN section below
# ---------------------------------------------------------------------------

def weights_init(m: nn.Module):
    '''DCGAN-style weight init (Radford et al., 2015).

    Why: GAN training is a saddle-point optimization between two networks
    that constantly move each other's loss landscape. Starting both nets
    from small, zero-centered weights (N(0, 0.02)) avoids one network
    dominating early purely due to init scale, which is a common cause of
    early, unrecoverable mode collapse.

    Uses isinstance (NOT a substring match on the class name) because many
    custom container classes in this notebook -- PGAN_ConvBlock,
    ModulatedConv2d, StyleConvBlock -- have "Conv" in their own name; a
    naive `"Conv" in classname` check would incorrectly try to initialize
    THOSE containers directly (they have no `.weight` of their own) instead
    of only their actual nn.Conv2d/nn.Linear/nn.BatchNorm2d leaf children.
    '''
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None:
            nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)) and getattr(m, "affine", False):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def show_images(images: torch.Tensor, title: str = "", nrow: int = 8, figsize=(8, 8), normalize=True):
    '''Display a batch of image tensors (B, C, H, W) in [-1, 1] or [0, 1] as a grid.'''
    images = images.detach().cpu()
    grid = vutils.make_grid(images, nrow=nrow, normalize=normalize, value_range=(-1, 1) if normalize else None, padding=2)
    plt.figure(figsize=figsize)
    plt.axis("off")
    plt.title(title)
    npimg = grid.numpy()
    if npimg.shape[0] == 1:
        plt.imshow(npimg[0], cmap="gray")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


def save_checkpoint(state: dict, path: str):
    '''Persist model/optimizer state so long training runs survive a Colab disconnect.'''
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def load_checkpoint(path: str, map_location=None):
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location or DEVICE)


class EMA:
    '''Exponential Moving Average of generator weights.

    Why: instantaneous generator weights oscillate a lot during adversarial
    training (they are chasing a moving discriminator). Averaging weights
    over time (as StyleGAN2 and BigGAN both do) gives a much smoother,
    higher-quality generator for *inference*, at zero extra training cost.
    '''
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


class AvgMeter:
    '''Running mean tracker for loss logging.'''
    def __init__(self):
        self.sum, self.n = 0.0, 0

    def update(self, val, k=1):
        self.sum += float(val) * k
        self.n += k

    @property
    def avg(self):
        return self.sum / max(self.n, 1)


def denorm(x: torch.Tensor) -> torch.Tensor:
    '''Map a tensor from [-1, 1] (tanh output range) back to [0, 1] for display/saving.'''
    return (x.clamp(-1, 1) + 1) / 2


CHECKPOINT_DIR = "checkpoints"
SAMPLES_DIR = "samples"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
print("Shared utilities ready.")


<a id="4"></a>
## 4. Progressive GAN -- Stable high-resolution generation via progressive growing

### 4.1 Problem Statement & Why this GAN

Karras et al. (2018) observed that training a deep GAN directly at high
resolution (1024x1024) is extremely unstable: early in training neither
network has learned coarse structure yet, and the discriminator can
trivially spot fine-detail artifacts, giving the generator no useful
gradient. **Progressive growing** fixes this by starting both networks at
a tiny 4x4 resolution, training to convergence, then **adding a new block**
that doubles resolution (4->8->16->...), fading it in smoothly with a
blend weight `alpha` so the new layers never see a "shocking" untrained
input. This is the **stable high-resolution generation** problem.

We grow 4 -> 8 -> 16 -> 32 on CIFAR-10 (32x32 is already CIFAR-10's native
resolution, so this section demonstrates the *mechanism* end-to-end
quickly; the real ProGAN paper grows to 1024x1024 over ~9 stages and
many GPU-days).

⚠️ **Warning -- Compute.** Even this 4-stage lightweight version trains
4 short sub-schedules; expect the longest single-GPU run in the "easy"
half of this notebook. Reduce `PGAN_EPOCHS_PER_STAGE` first if short on time.

### 4.2 Data Collection & Preparation

In [ ]:
PGAN_FINAL_SIZE = 32                    # 4 -> 8 -> 16 -> 32
PGAN_STAGES = [4, 8, 16, 32]
PGAN_NZ = 128
PGAN_BATCH_SIZES = {4: 128, 8: 128, 16: 64, 32: 32}   # shrink batch as resolution (and memory) grows
PGAN_EPOCHS_PER_STAGE = 2                # x2 substages each (fade-in + stabilize) -- see training loop

try:
    pgan_base = torchvision.datasets.CIFAR10(root="data", train=True, download=True,
                                              transform=T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)]))
    print(f"Loaded real CIFAR-10: {len(pgan_base)} images")
except Exception as e:
    print(f"CIFAR-10 download failed ({e}); falling back to synthetic textures.")
    rng = np.random.default_rng(SEED)
    imgs = torch.rand(2000, 3, PGAN_FINAL_SIZE, PGAN_FINAL_SIZE) * 2 - 1
    pgan_base = TensorDataset(imgs, torch.zeros(2000, dtype=torch.long))

def pgan_loader_for_size(size):
    '''Build a DataLoader that resizes every image to the CURRENT stage's
    resolution on the fly -- this is what lets one dataset feed every
    growth stage without re-downloading/re-processing.'''
    resize = T.Resize(size)
    class Resized(Dataset):
        def __len__(self_): return len(pgan_base)
        def __getitem__(self_, i):
            x, y = pgan_base[i]
            return resize(x), y
    bs = PGAN_BATCH_SIZES[size]
    return DataLoader(Resized(), batch_size=bs, shuffle=True, drop_last=True)

show_images(next(iter(pgan_loader_for_size(32)))[0][:32], title="Real data (final 32x32 stage)", nrow=8)


### 4.3 Model Architecture

In [ ]:
def pgan_pixel_norm(x, eps=1e-8):
    '''Pixelwise feature normalization (ProGAN's replacement for BatchNorm
    in the generator): normalize each pixel's channel vector to unit norm.
    Why: prevents signal magnitudes from escalating uncontrollably across
    the many blocks of a progressively-growing generator, without the
    batch-statistics coupling of BatchNorm (which the paper found hurt
    unbounded stochastic growing).'''
    return x / torch.sqrt(x.pow(2).mean(dim=1, keepdim=True) + eps)


class PGAN_ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.c1 = nn.Conv2d(in_c, out_c, 3, 1, 1)
        self.c2 = nn.Conv2d(out_c, out_c, 3, 1, 1)
        self.act = nn.LeakyReLU(0.2, True)

    def forward(self, x):
        x = pgan_pixel_norm(self.act(self.c1(x)))
        x = pgan_pixel_norm(self.act(self.c2(x)))
        return x


class PGAN_Generator(nn.Module):
    '''Progressively-growable generator. `to_rgb` layers exist for every
    stage; `forward(z, stage, alpha)` runs the network up to `stage` and,
    if alpha < 1, cross-fades between the *upsampled previous-stage* RGB
    output and the *new stage's* RGB output -- the core trick that avoids
    destabilizing shocks when a new block is introduced mid-training.'''
    def __init__(self, nz=PGAN_NZ, stages=PGAN_STAGES, base_ch=256):
        super().__init__()
        self.stages = stages
        chs = {s: max(32, base_ch // (2 ** i)) for i, s in enumerate(stages)}   # shrink channels as resolution grows (standard ProGAN schedule)
        self.chs = chs
        self.init = nn.Sequential(
            nn.ConvTranspose2d(nz, chs[stages[0]], stages[0], 1, 0), nn.LeakyReLU(0.2, True),
        )  # z -> 4x4 base feature map
        self.blocks = nn.ModuleList()   # one growth block per stage AFTER the first
        self.to_rgb = nn.ModuleList([nn.Conv2d(chs[stages[0]], 3, 1)])
        in_c = chs[stages[0]]
        for s in stages[1:]:
            out_c = chs[s]
            self.blocks.append(PGAN_ConvBlock(in_c, out_c))
            self.to_rgb.append(nn.Conv2d(out_c, 3, 1))
            in_c = out_c

    def forward(self, z, stage_idx: int, alpha: float = 1.0):
        x = pgan_pixel_norm(self.init(z.view(z.size(0), -1, 1, 1)))
        if stage_idx == 0:
            return torch.tanh(self.to_rgb[0](x))
        prev_rgb = None
        for i in range(stage_idx):
            if i == stage_idx - 1:
                prev_rgb = F.interpolate(self.to_rgb[i](x), scale_factor=2, mode="nearest")
            x = F.interpolate(x, scale_factor=2, mode="nearest")
            x = self.blocks[i](x)
        new_rgb = self.to_rgb[stage_idx](x)
        if alpha >= 1.0 or prev_rgb is None:
            return torch.tanh(new_rgb)
        return torch.tanh(alpha * new_rgb + (1 - alpha) * prev_rgb)   # fade-in blend


class PGAN_Discriminator(nn.Module):
    '''Mirror of the generator: `from_rgb` layers per stage, and the SAME
    fade-in blending on the way IN so the discriminator also sees a smooth
    transition, not a sudden new block.'''
    def __init__(self, stages=PGAN_STAGES, base_ch=256):
        super().__init__()
        self.stages = stages
        chs = {s: max(32, base_ch // (2 ** i)) for i, s in enumerate(stages)}
        self.from_rgb = nn.ModuleList([nn.Conv2d(3, chs[s], 1) for s in stages])
        self.blocks = nn.ModuleList()
        rev = list(reversed(stages))
        for i in range(len(rev) - 1):
            in_c, out_c = chs[rev[i]], chs[rev[i + 1]]
            self.blocks.append(nn.Sequential(
                nn.Conv2d(in_c, in_c, 3, 1, 1), nn.LeakyReLU(0.2, True),
                nn.Conv2d(in_c, out_c, 3, 1, 1), nn.LeakyReLU(0.2, True),
            ))
        self.final = nn.Sequential(
            nn.Conv2d(chs[stages[0]], chs[stages[0]], 3, 1, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(chs[stages[0]], 1, stages[0], 1, 0),
        )

    def forward(self, img, stage_idx: int, alpha: float = 1.0):
        x = F.leaky_relu(self.from_rgb[stage_idx](img), 0.2)
        if stage_idx > 0 and alpha < 1.0:
            downsampled = F.avg_pool2d(img, 2)
            skip = F.leaky_relu(self.from_rgb[stage_idx - 1](downsampled), 0.2)
        for i, block_i in enumerate(range(stage_idx - 1, -1, -1)):
            x = self.blocks[len(self.stages) - 2 - block_i](x)
            x = F.avg_pool2d(x, 2)
            if i == 0 and stage_idx > 0 and alpha < 1.0:
                x = alpha * x + (1 - alpha) * skip
        return self.final(x).view(-1)


G_pgan = PGAN_Generator().to(DEVICE).apply(weights_init)
D_pgan = PGAN_Discriminator().to(DEVICE).apply(weights_init)
print(f"ProGAN stages: {PGAN_STAGES}")


### 4.4 Loss Functions & Optimizers

In [ ]:
# WGAN-GP loss: ProGAN's original paper uses WGAN-GP (Gulrajani et al.)
# rather than the vanilla BCE minimax loss, because the gradient penalty
# keeps the discriminator 1-Lipschitz, which is empirically far more
# stable across the many growth-stage transitions than BCE.
def pgan_gradient_penalty(D, real, fake, stage_idx, alpha, device=DEVICE):
    b = real.size(0)
    eps = torch.rand(b, 1, 1, 1, device=device)
    interp = (eps * real + (1 - eps) * fake).requires_grad_(True)
    d_interp = D(interp, stage_idx, alpha)
    grad = torch.autograd.grad(outputs=d_interp, inputs=interp,
                                grad_outputs=torch.ones_like(d_interp),
                                create_graph=True, retain_graph=True)[0]
    grad = grad.view(b, -1)
    return ((grad.norm(2, dim=1) - 1) ** 2).mean()

PGAN_LAMBDA_GP = 10.0
PGAN_DRIFT = 1e-3   # small penalty keeping D's outputs near 0 (prevents unbounded drift under WGAN's unconstrained loss)
pgan_opt_G = torch.optim.Adam(G_pgan.parameters(), lr=1e-3, betas=(0.0, 0.99))
pgan_opt_D = torch.optim.Adam(D_pgan.parameters(), lr=1e-3, betas=(0.0, 0.99))


### 4.5 Training Loop

In [ ]:
# ---------------------------------------------------------------------------
# 4.5a Lightweight from-scratch progressive training
# ---------------------------------------------------------------------------
# Schedule per stage: first PGAN_EPOCHS_PER_STAGE epochs FADE IN (alpha: 0->1),
# then PGAN_EPOCHS_PER_STAGE epochs STABILIZE (alpha=1). Stage 0 (4x4) has no
# fade-in (nothing to fade in from).
pgan_fixed_noise = torch.randn(16, PGAN_NZ, device=DEVICE)

for stage_idx, size in enumerate(PGAN_STAGES):
    loader = pgan_loader_for_size(size)
    n_batches_total = len(loader) * PGAN_EPOCHS_PER_STAGE
    substages = [("stabilize", PGAN_EPOCHS_PER_STAGE)] if stage_idx == 0 else \
                [("fade-in", PGAN_EPOCHS_PER_STAGE), ("stabilize", PGAN_EPOCHS_PER_STAGE)]

    for phase, n_epochs in substages:
        for epoch in range(n_epochs):
            pbar = tqdm(loader, desc=f"[ProGAN] {size}x{size} {phase} epoch {epoch+1}/{n_epochs}")
            for step, (real, _) in enumerate(pbar):
                real = real.to(DEVICE)
                b = real.size(0)
                if phase == "fade-in":
                    alpha = min(1.0, (epoch * len(loader) + step) / max(1, n_batches_total))
                else:
                    alpha = 1.0

                # ---- Discriminator (WGAN-GP critic) ----
                D_pgan.zero_grad(set_to_none=True)
                z = torch.randn(b, PGAN_NZ, device=DEVICE)
                fake = G_pgan(z, stage_idx, alpha)
                d_real = D_pgan(real, stage_idx, alpha)
                d_fake = D_pgan(fake.detach(), stage_idx, alpha)
                gp = pgan_gradient_penalty(D_pgan, real, fake.detach(), stage_idx, alpha)
                loss_d = d_fake.mean() - d_real.mean() + PGAN_LAMBDA_GP * gp + PGAN_DRIFT * (d_real ** 2).mean()
                loss_d.backward()
                pgan_opt_D.step()

                # ---- Generator ----
                G_pgan.zero_grad(set_to_none=True)
                d_fake_g = D_pgan(fake, stage_idx, alpha)
                loss_g = -d_fake_g.mean()
                loss_g.backward()
                pgan_opt_G.step()

                pbar.set_postfix(alpha=f"{alpha:.2f}", loss_D=f"{loss_d.item():.2f}", loss_G=f"{loss_g.item():.2f}")

        with torch.no_grad():
            G_pgan.eval()
            sample = G_pgan(pgan_fixed_noise, stage_idx, alpha=1.0)
            G_pgan.train()
        show_images(sample, title=f"ProGAN samples -- stage {size}x{size} ({phase} done)", nrow=4)

    save_checkpoint({"G": G_pgan.state_dict(), "D": D_pgan.state_dict(), "stage_idx": stage_idx},
                     f"{CHECKPOINT_DIR}/pgan_stage_{size}.pt")

print("Progressive GAN growth complete: 4x4 -> 32x32.")


In [ ]:
# ---------------------------------------------------------------------------
# 4.5b Loading strong pretrained weights (inference-only mode)
# ---------------------------------------------------------------------------
# The official NVIDIA ProGAN checkpoints (e.g. CelebA-HQ 1024x1024) are
# distributed as pickled TensorFlow graphs / torch_utils-dependent .pkl
# files via NVlabs' repos, and Facebook Research's pytorch_GAN_zoo package
# offers a torch.hub-loadable PyTorch port. We attempt that here; if there
# is no network access this simply reports so and the notebook continues
# using the from-scratch model trained above.
def try_load_pretrained_progan():
    try:
        pgan_pretrained = torch.hub.load(
            "facebookresearch/pytorch_GAN_zoo:hub", "PGAN",
            model_name="celebAHQ-512", pretrained=True, useGPU=(DEVICE.type == "cuda"),
        )
        print("Loaded official pretrained Progressive GAN (CelebA-HQ 512x512).")
        return pgan_pretrained
    except Exception as e:
        print(f"Pretrained ProGAN unavailable in this environment ({e}). "
              f"Using the from-scratch model trained above instead.")
        return None

pgan_pretrained = try_load_pretrained_progan()


### 4.6 Inference / Generation Function

In [ ]:
@torch.no_grad()
def pgan_generate(n_samples: int = 16, seed: int = None, use_pretrained: bool = False) -> torch.Tensor:
    '''Unified generation function: uses the official pretrained model if it
    was loaded successfully AND the caller asks for it, otherwise falls
    back to our from-scratch model at the final trained stage (32x32).'''
    if seed is not None:
        torch.manual_seed(seed)
    if use_pretrained and pgan_pretrained is not None:
        noise, _ = pgan_pretrained.buildNoiseData(n_samples)
        with torch.no_grad():
            imgs = pgan_pretrained.test(noise)
        return imgs
    G_pgan.eval()
    z = torch.randn(n_samples, PGAN_NZ, device=DEVICE)
    imgs = G_pgan(z, stage_idx=len(PGAN_STAGES) - 1, alpha=1.0)
    G_pgan.train()
    return imgs

show_images(pgan_generate(16), title="ProGAN inference (from-scratch, 32x32)", nrow=4)


### 4.7 Visualization & Results

In [ ]:
# Growth progression: show the SAME latent vectors decoded at every
# resolution stage side by side -- visualizes exactly what "progressive"
# growing means for a fixed identity in latent space.
with torch.no_grad():
    G_pgan.eval()
    z_demo = torch.randn(4, PGAN_NZ, device=DEVICE)
    fig, axes = plt.subplots(4, len(PGAN_STAGES), figsize=(2.2 * len(PGAN_STAGES), 8))
    for col, size in enumerate(PGAN_STAGES):
        imgs = denorm(G_pgan(z_demo, stage_idx=col, alpha=1.0)).cpu()
        for row in range(4):
            ax = axes[row, col]
            ax.imshow(imgs[row].permute(1, 2, 0).numpy())
            ax.axis("off")
            if row == 0:
                ax.set_title(f"{size}x{size}")
    plt.suptitle("Same latent code decoded at each growth stage")
    plt.tight_layout(); plt.show()
    G_pgan.train()


### 4.8 User Input → Output Demo

In [ ]:
N_SAMPLES_TO_GENERATE = 9
RANDOM_SEED = 123
USE_PRETRAINED_IF_AVAILABLE = True

demo = pgan_generate(N_SAMPLES_TO_GENERATE, seed=RANDOM_SEED, use_pretrained=USE_PRETRAINED_IF_AVAILABLE)
show_images(demo, title="ProGAN user demo", nrow=3)


### 4.9 Key Insights & Limitations of this GAN

- 💡 **Insight.** Progressive growing is fundamentally a **curriculum
  learning** strategy for GANs: solve the easy, low-frequency coarse-shape
  problem first (4x4), then gradually add the harder, high-frequency
  detail problem, instead of asking a randomly initialized network to
  solve both simultaneously.
- 🚀 **Pro Tip.** The `alpha` fade-in schedule is the single most important
  hyperparameter here -- fading too fast reintroduces the instability
  progressive growing was designed to avoid; too slow wastes compute.
- ⚠️ **Warning.** ProGAN has no built-in mechanism for *disentangled,
  controllable* style (e.g. "keep pose, change hair color") -- that
  capability is exactly what StyleGAN (next section) adds on top of this
  same progressive-growing backbone.

**This GAN solved the problem of: stable training at (and generation of)
high resolutions, by growing both networks gradually instead of training
a deep net at full resolution from a random initialization.**